# Dagbani ASR Phase 1 — Data Audit and Comparable Baselines

This is the **CPU-first evidence notebook**. It discovers real corpora, creates
the canonical manifest, audits audio/transcript quality and split leakage, then
optionally benchmarks Whisper models on one frozen WAXAL evaluation set.

Safety rules:

- Missing data is reported as `unavailable`, `gated`, or `rejected`; it is never
  replaced with synthetic audio.
- `transcript_raw` is immutable. Evaluation normalization only applies Unicode
  NFC, lowercase, whitespace cleanup, and punctuation removal.
- WAXAL's official train/validation/test split is preserved. The test split is
  sealed and is never used for training or pseudo-labelling.
- Run `quick` mode first. Use `full` only after inspecting the source report.
- Keep the accelerator **off** until the optional baseline section.

## 1. Install a bounded, Kaggle-compatible environment

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_DEPS = True
REQUIRED = {
    "datasets": "datasets[audio]>=3.2,<5",
    "transformers": "transformers>=4.48,<6",
    "accelerate": "accelerate>=1.2,<3",
    "huggingface_hub": "huggingface_hub>=0.27,<2",
    "jiwer": "jiwer>=3.0,<5",
    "soundfile": "soundfile>=0.12,<1",
    "librosa": "librosa>=0.10,<1",
    "pyarrow": "pyarrow>=17,<25",
    "pandas": "pandas>=2.1,<4",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if INSTALL_DEPS and missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Environment ready. PyTorch is intentionally not replaced.")

## 2. Configuration — quick mode is intentionally the default

In [ ]:
from __future__ import annotations

import dataclasses
import gzip
import hashlib
import importlib.metadata
import io
import json
import os
import platform
import re
import string
import subprocess
import textwrap
import time
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator

import numpy as np
import pandas as pd
import requests
import soundfile as sf
from IPython.display import display


def kaggle_secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


@dataclass
class AuditConfig:
    audit_mode: str = "quick"  # quick | full
    work_dir: str = "/kaggle/working/dagbani_asr_phase1"
    hf_token: str | None = field(default_factory=lambda: kaggle_secret("HF_TOKEN"))
    manifest_repo_id: str = os.getenv("DAGBANI_MANIFEST_REPO", "")
    publish_artifacts: bool = False
    run_baselines: bool = False
    baseline_limit: int | None = 256
    metadata_rows_per_split: int = 2_000
    audio_rows_per_split: int = 100
    unlabeled_metadata_rows: int = 5_000
    unlabeled_audio_rows: int = 200
    spell4wiki_quick_limit: int = 500
    seed: int = 42
    min_duration_s: float = 0.5
    max_duration_s: float = 30.0
    min_words_per_s: float = 0.25
    max_words_per_s: float = 8.0
    waxal_benchmark_min_duration_s: float = 1.5
    waxal_benchmark_min_words_per_s: float = 4.0
    common_voice_root: str = os.getenv("COMMON_VOICE_DAG_ROOT", "")
    local_manifest_paths: tuple[str, ...] = ()
    # Empty means every discoverable source. For the first supervised
    # milestone use ("waxal_dag_asr",) and audit Bible data later.
    enabled_sources: tuple[str, ...] = ()
    allow_network_audio_downloads: bool = False

    def validate(self) -> None:
        if self.audit_mode not in {"quick", "full"}:
            raise ValueError("audit_mode must be 'quick' or 'full'")
        if self.publish_artifacts and not self.manifest_repo_id:
            raise ValueError("Set DAGBANI_MANIFEST_REPO before publishing")
        if self.publish_artifacts and not self.hf_token:
            raise ValueError("HF_TOKEN is required for private publishing")

    @property
    def metadata_limit(self) -> int | None:
        return None if self.audit_mode == "full" else self.metadata_rows_per_split

    @property
    def audio_limit(self) -> int | None:
        return None if self.audit_mode == "full" else self.audio_rows_per_split


cfg = AuditConfig()
cfg.validate()
WORK_DIR = Path(cfg.work_dir)
ARTIFACT_DIR = WORK_DIR / "artifacts"
CACHE_DIR = WORK_DIR / "hf_cache"
for directory in (WORK_DIR, ARTIFACT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")
print(json.dumps(asdict(cfg) | {"hf_token": "<set>" if cfg.hf_token else "<missing>"}, indent=2))

## 3. Canonical text and audio helpers

In [ ]:
CANONICAL_COLUMNS = [
    "sample_id", "source", "source_version", "audio_locator",
    "transcript_raw", "transcript_eval", "speaker_id", "duration_s",
    "domain", "split", "license", "audio_hash", "is_pseudo",
    "pseudo_confidence",
]
DAGBANI_GLYPHS = set("ɛɔŋɣʒƐƆŊƔƷ")
PUNCT_TRANSLATION = str.maketrans({char: " " for char in string.punctuation + "“”‘’…–—"})


def normalize_eval(text: Any) -> str:
    '''Minimal evaluation-only normalization; never transliterate letters.'''
    value = unicodedata.normalize("NFC", str(text or "")).lower()
    value = value.translate(PUNCT_TRANSLATION)
    return " ".join(value.split())


def strict_text(text: Any) -> str:
    return unicodedata.normalize("NFC", str(text or "")).strip()


def unexpected_characters(text: str) -> list[str]:
    allowed_categories = {"Ll", "Lu", "Lt", "Lm", "Lo", "Mn", "Mc", "Nd", "Zs", "Po", "Pd"}
    return sorted({ch for ch in text if unicodedata.category(ch) not in allowed_categories and ch not in "\n\t"})


def stable_split(group_id: str, train: int = 80, validation: int = 10) -> str:
    bucket = int(hashlib.sha1(group_id.encode("utf-8")).hexdigest()[:8], 16) % 100
    if bucket < train:
        return "train"
    if bucket < train + validation:
        return "validation"
    return "test"


def bible_group(record: dict[str, Any], locator: str) -> str | None:
    for key in ("book_chapter", "chapter", "book", "group_id"):
        value = record.get(key)
        if value not in (None, ""):
            book = record.get("book", "book")
            return f"{book}:{value}"
    match = re.search(r"(?i)(genesis|exodus|leviticus|numbers|deuteronomy|[1-3]?\s*[a-z]+)[_\-/ ]*(\d{1,3})", locator)
    return f"{match.group(1).lower()}:{match.group(2)}" if match else None


def audio_payload(value: Any) -> tuple[np.ndarray, int, bytes | None]:
    '''Decode HF Audio dicts, torchcodec decoders, local paths, or byte payloads.'''
    raw_bytes = None
    if hasattr(value, "get_all_samples"):
        samples = value.get_all_samples()
        array = samples.data.detach().cpu().numpy()
        sample_rate = int(samples.sample_rate)
    elif isinstance(value, dict) and value.get("array") is not None:
        array = np.asarray(value["array"], dtype=np.float32)
        sample_rate = int(value.get("sampling_rate") or 16_000)
        raw_bytes = value.get("bytes")
    else:
        path = None
        if isinstance(value, dict):
            raw_bytes = value.get("bytes")
            path = value.get("path")
        elif isinstance(value, (bytes, bytearray)):
            raw_bytes = bytes(value)
        elif isinstance(value, (str, os.PathLike)):
            path = str(value)
        if raw_bytes:
            array, sample_rate = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=False)
        elif path and re.match(r"https?://", path):
            if not cfg.allow_network_audio_downloads:
                raise PermissionError("Network audio download is disabled")
            response = requests.get(path, timeout=60)
            response.raise_for_status()
            raw_bytes = response.content
            array, sample_rate = sf.read(io.BytesIO(raw_bytes), dtype="float32", always_2d=False)
        elif path:
            array, sample_rate = sf.read(path, dtype="float32", always_2d=False)
        else:
            raise ValueError("Unsupported or empty audio payload")
    array = np.asarray(array, dtype=np.float32)
    if array.ndim == 2:
        array = array.mean(axis=0 if array.shape[0] <= 8 else 1)
    return array.reshape(-1), int(sample_rate), raw_bytes


def canonical_audio(array: np.ndarray, sample_rate: int) -> np.ndarray:
    import librosa
    if sample_rate != 16_000:
        array = librosa.resample(array, orig_sr=sample_rate, target_sr=16_000)
    peak = float(np.max(np.abs(array))) if len(array) else 0.0
    if peak > 1.0:
        array = array / peak
    return np.clip(array, -1.0, 1.0).astype(np.float32)


def exact_audio_hash(array: np.ndarray) -> str:
    pcm16 = np.round(np.clip(array, -1, 1) * 32767).astype("<i2")
    return hashlib.sha256(pcm16.tobytes()).hexdigest()


def spectral_fingerprint(array: np.ndarray) -> str:
    '''A compact codec-tolerant log-mel fingerprint for near-duplicate grouping.'''
    import librosa
    if len(array) < 800:
        return ""
    mel = librosa.feature.melspectrogram(y=array, sr=16_000, n_fft=400, hop_length=320, n_mels=24)
    log_mel = librosa.power_to_db(mel + 1e-10, ref=np.max)
    old_x = np.linspace(0.0, 1.0, log_mel.shape[1])
    new_x = np.linspace(0.0, 1.0, 32)
    resized = np.vstack([np.interp(new_x, old_x, row) for row in log_mel])
    quantized = np.clip(np.round((resized + 80.0) * 3.0), 0, 255).astype(np.uint8)
    return hashlib.sha256(quantized.tobytes()).hexdigest()


assert normalize_eval("  N NYƐLA—BƐ!  ") == "n nyɛla bɛ"
assert strict_text("e\u0301") == "é"
print("Canonical schema and conservative normalization checks passed.")

## 4. Source registry and availability discovery

In [ ]:
from datasets import Audio, get_dataset_config_info, get_dataset_config_names, load_dataset


@dataclass(frozen=True)
class SourceSpec:
    name: str
    repo_id: str
    config_name: str | None
    domain: str
    expected_license: str
    required: bool = False
    text_fields: tuple[str, ...] = ("transcription", "sentence", "text", "transcript")
    audio_fields: tuple[str, ...] = ("audio", "speech")
    speaker_fields: tuple[str, ...] = ("speaker_id", "client_id", "speaker", "user_id")
    id_fields: tuple[str, ...] = ("id", "sample_id", "path", "file", "filename")


HF_SOURCES = [
    SourceSpec("waxal_dag_asr", "google/WaxalNLP", "dag_asr", "spontaneous", "cc-by-4.0", True),
    SourceSpec("dagbani_bible", "ghananlpcommunity/dagbani-bible-audio-text-tts", None, "bible_read", "cc-by-nc-4.0"),
    SourceSpec("ghana_speech_dag", "ghananlpcommunity/ghana-speech", "Dagbani_dag", "read_speech", "unknown"),
    SourceSpec("navigation_dagbani", "ghananlpcommunity/navigation-corpus-dagbani-speech", None, "navigation", "cc-by-nc-4.0"),
    SourceSpec("health_unicef_dagbani", "ghananlpcommunity/ghana-nlp-health-UNICEF-asr-dagbani", None, "health", "unknown"),
    SourceSpec("youth_conversations_dagbani", "ghananlpcommunity/youth-conversations-dag", None, "conversation", "unknown"),
]


def split_info_value(value: Any, field_name: str) -> int | None:
    '''Read Hugging Face split metadata across datasets library versions.'''
    if isinstance(value, dict):
        raw = value.get(field_name)
    else:
        raw = getattr(value, field_name, None)
    return int(raw) if raw is not None else None


def discover_hf_source(spec: SourceSpec) -> dict[str, Any]:
    started = time.time()
    report = asdict(spec) | {"status": "unavailable", "splits": {}, "reported_license": ""}
    try:
        configs = get_dataset_config_names(spec.repo_id, token=cfg.hf_token)
        config_name = spec.config_name
        if config_name and config_name not in configs:
            raise ValueError(f"config {config_name!r} not found; available={configs[:20]}")
        if config_name is None and len(configs) == 1:
            config_name = configs[0]
        info = get_dataset_config_info(spec.repo_id, config_name, token=cfg.hf_token)
        report["resolved_config"] = config_name
        report["reported_license"] = str(getattr(info, "license", "") or "")
        report["splits"] = {
            name: {
                "num_examples": split_info_value(value, "num_examples"),
                "num_bytes": split_info_value(value, "num_bytes"),
            }
            for name, value in (getattr(info, "splits", {}) or {}).items()
        }
        explicit_license = report["reported_license"] or spec.expected_license
        report["status"] = "available" if explicit_license != "unknown" else "gated"
        report["license"] = explicit_license
    except Exception as exc:
        report["error"] = f"{type(exc).__name__}: {exc}"
    report["elapsed_s"] = round(time.time() - started, 2)
    return report


source_reports = [discover_hf_source(spec) for spec in HF_SOURCES]

cv_root = Path(cfg.common_voice_root) if cfg.common_voice_root else None
cv_report = {
    "name": "common_voice_25_dag",
    "status": "available" if cv_root and (cv_root / "validated.tsv").exists() else "gated",
    "domain": "scripted_read",
    "license": "cc0-1.0",
    "locator": str(cv_root or "Set COMMON_VOICE_DAG_ROOT after accepting Mozilla terms"),
}
source_reports.append(cv_report)

local_reports = []
for raw_path in cfg.local_manifest_paths:
    path = Path(raw_path)
    local_reports.append({
        "name": f"local:{path.stem}", "status": "available" if path.exists() else "unavailable",
        "domain": "user_supplied", "license": "gated", "locator": str(path),
    })
source_reports.extend(local_reports)
source_status_df = pd.DataFrame(source_reports)
display(source_status_df[[column for column in ["name", "status", "domain", "license", "locator", "error"] if column in source_status_df]])
print(source_status_df[[column for column in ["name", "status", "resolved_config", "license", "error"] if column in source_status_df]].to_string(index=False))

## 5. Bounded adapters build the canonical manifest

In [ ]:
def first_field(record: dict[str, Any], candidates: Iterable[str]) -> str | None:
    return next((name for name in candidates if name in record), None)


def source_split(spec: SourceSpec, original_split: str, record: dict[str, Any], locator: str, speaker: str) -> str:
    if spec.name == "waxal_dag_asr":
        return original_split
    if "bible" in spec.name:
        group = bible_group(record, locator)
        return stable_split(group) if group else "external_test"
    if speaker and speaker != "unknown":
        return stable_split(f"{spec.name}:{speaker}")
    return "external_test"


def audio_locator(spec: SourceSpec, split: str, sample_id: str, value: Any) -> str:
    if isinstance(value, dict) and value.get("path"):
        return str(value["path"])
    if isinstance(value, (str, os.PathLike)):
        return str(value)
    return f"hf://{spec.repo_id}/{spec.config_name or 'default'}/{split}/{sample_id}"


def canonical_record(
    spec: SourceSpec,
    original_split: str,
    record: dict[str, Any],
    row_index: int,
    decode_audio: bool,
) -> tuple[dict[str, Any] | None, dict[str, Any] | None, str | None]:
    text_field = first_field(record, spec.text_fields)
    audio_field = first_field(record, spec.audio_fields)
    id_field = first_field(record, spec.id_fields)
    speaker_field = first_field(record, spec.speaker_fields)
    sample_id = strict_text(record.get(id_field)) if id_field else f"{original_split}-{row_index:09d}"
    text = strict_text(record.get(text_field)) if text_field else ""
    speaker = strict_text(record.get(speaker_field)) if speaker_field else "unknown"
    speaker_id = f"{spec.name}:{speaker}" if speaker and speaker != "unknown" else "unknown"
    value = record.get(audio_field) if audio_field else None
    locator = audio_locator(spec, original_split, sample_id, value)
    split = source_split(spec, original_split, record, locator, speaker)
    split_policy = "official" if spec.name == "waxal_dag_asr" else "speaker_or_group"
    if "bible" in spec.name and split == "external_test":
        # The current 16-word Bible corpus exposes only audio/text/duration.
        # Keep adjacent aligned segments together in large contiguous blocks
        # when explicit book/chapter metadata is absent.
        group = f"contiguous-block-{row_index // 500:06d}"
        split = stable_split(f"{spec.name}:{group}")
        split_policy = "contiguous_500_row_fallback"
    license_name = spec.expected_license
    declared_duration = record.get("duration_s", record.get("duration", np.nan))
    try:
        declared_duration = float(declared_duration)
    except (TypeError, ValueError):
        declared_duration = np.nan
    base = {
        "sample_id": sample_id,
        "source": spec.name,
        "source_version": spec.config_name or "default",
        "audio_locator": locator,
        "transcript_raw": text,
        "transcript_eval": normalize_eval(text),
        # Speaker identifiers are source-local. Namespace them so numeric or
        # short IDs from unrelated corpora cannot create false leakage.
        "speaker_id": speaker_id,
        "speaker_id_raw": speaker or "unknown",
        "duration_s": declared_duration,
        "domain": spec.domain,
        "split": split,
        "origin_split": original_split,
        "split_policy": split_policy,
        "license": license_name,
        "audio_hash": "",
        "is_pseudo": False,
        "pseudo_confidence": np.nan,
    }
    is_unlabeled = original_split == "unlabeled"
    if license_name == "unknown":
        return None, base | {"reason": "license_not_verified"}, None
    if not audio_field:
        return None, base | {"reason": "audio_field_missing"}, None
    if not text and not is_unlabeled:
        return None, base | {"reason": "transcript_missing"}, None
    if unexpected_characters(text):
        base["unexpected_characters"] = "".join(unexpected_characters(text))
    if not decode_audio:
        return base, None, None
    try:
        array, sample_rate, _ = audio_payload(value)
        array = canonical_audio(array, sample_rate)
        duration = len(array) / 16_000
        base["duration_s"] = round(duration, 4)
        base["audio_hash"] = exact_audio_hash(array)
        near_hash = spectral_fingerprint(array)
        words_per_s = len(base["transcript_eval"].split()) / max(duration, 1e-6) if text else np.nan
        if duration < cfg.min_duration_s or duration > cfg.max_duration_s:
            return None, base | {"reason": "duration_out_of_range", "words_per_s": words_per_s}, near_hash
        if text and not (cfg.min_words_per_s <= words_per_s <= cfg.max_words_per_s):
            return None, base | {"reason": "speech_rate_out_of_range", "words_per_s": words_per_s}, near_hash
        base["words_per_s"] = words_per_s
        return base, None, near_hash
    except Exception as exc:
        return None, base | {"reason": "audio_decode_failed", "error": f"{type(exc).__name__}: {exc}"}, None


def scan_hf_source(spec: SourceSpec, report: dict[str, Any]) -> tuple[list[dict], list[dict], list[dict]]:
    accepted, rejected, near = [], [], []
    if cfg.enabled_sources and spec.name not in cfg.enabled_sources:
        print(f"{spec.name}: skipped by enabled_sources")
        return accepted, rejected, near
    if report.get("status") != "available":
        return accepted, rejected, near
    config_name = report.get("resolved_config", spec.config_name)
    resolved_spec = dataclasses.replace(
        spec,
        config_name=config_name,
        expected_license=str(report.get("license") or spec.expected_license),
    )
    split_names = list((report.get("splits") or {}).keys()) or ["train"]
    for split_name in split_names:
        split_started = time.time()
        accepted_before, rejected_before = len(accepted), len(rejected)
        stream = load_dataset(spec.repo_id, config_name, split=split_name, streaming=True, token=cfg.hf_token)
        # Full mode decodes every supervised row, but deliberately samples the
        # very large unlabeled partition. Pseudo-labelling streams it later.
        limit = cfg.unlabeled_metadata_rows if split_name == "unlabeled" else cfg.metadata_limit
        audio_limit = cfg.unlabeled_audio_rows if split_name == "unlabeled" else cfg.audio_limit
        reported_rows = ((report.get("splits") or {}).get(split_name) or {}).get("num_examples")
        target_rows = min(limit, reported_rows) if limit is not None and reported_rows else (limit or reported_rows)
        for index, record in enumerate(stream):
            if limit is not None and index >= limit:
                break
            decode = audio_limit is None or index < audio_limit
            good, bad, fingerprint = canonical_record(resolved_spec, split_name, record, index, decode)
            if good:
                accepted.append(good)
                if fingerprint:
                    near.append({"sample_id": good["sample_id"], "source": spec.name, "near_hash": fingerprint})
            if bad:
                rejected.append(bad)
            processed = index + 1
            if processed % 250 == 0:
                elapsed = max(time.time() - split_started, 1e-6)
                rows_per_s = processed / elapsed
                eta_s = ((target_rows - processed) / rows_per_s) if target_rows and rows_per_s else None
                eta_text = f"{eta_s / 60:.1f}m" if eta_s is not None else "unknown"
                print(
                    f"{spec.name}/{split_name}: {processed:,}/{target_rows or '?'} rows "
                    f"({rows_per_s:.2f} rows/s, ETA {eta_text})"
                )
        print(
            f"{spec.name}/{split_name}: accepted={len(accepted) - accepted_before:,}, "
            f"rejected={len(rejected) - rejected_before:,}, elapsed={(time.time() - split_started) / 60:.1f}m"
        )
    return accepted, rejected, near


accepted_rows, rejected_rows, near_hash_rows = [], [], []
report_by_name = {report["name"]: report for report in source_reports}
for source_spec in HF_SOURCES:
    good, bad, near = scan_hf_source(source_spec, report_by_name[source_spec.name])
    accepted_rows.extend(good)
    rejected_rows.extend(bad)
    near_hash_rows.extend(near)
print(f"HF scan complete: {len(accepted_rows):,} accepted metadata rows, {len(rejected_rows):,} rejected rows")

### Optional Common Voice, local-manifest, and Spell4Wiki adapters

In [ ]:
def scan_common_voice(root: Path) -> tuple[list[dict], list[dict]]:
    accepted, rejected = [], []
    for split_name, filename in (("train", "train.tsv"), ("validation", "dev.tsv"), ("test", "test.tsv")):
        tsv = root / filename
        if not tsv.exists():
            continue
        table = pd.read_csv(tsv, sep="\t")
        limit = cfg.metadata_limit or len(table)
        for index, row in table.head(limit).iterrows():
            clip = root / "clips" / str(row["path"])
            record = {
                "id": str(row["path"]), "audio": str(clip), "sentence": row.get("sentence", ""),
                "client_id": row.get("client_id", "unknown"),
            }
            spec = SourceSpec("common_voice_25_dag", "local", None, "scripted_read", "cc0-1.0")
            good, bad, _ = canonical_record(spec, split_name, record, int(index), cfg.audit_mode == "full")
            (accepted if good else rejected).append(good or bad)
    return accepted, rejected


if cv_root and (cv_root / "validated.tsv").exists():
    good, bad = scan_common_voice(cv_root)
    accepted_rows.extend(good)
    rejected_rows.extend(bad)


def scan_local_manifest(path: Path) -> tuple[list[dict], list[dict]]:
    table = pd.read_json(path, lines=True) if path.suffix.lower() in {".json", ".jsonl"} else pd.read_csv(path, sep="\t" if path.suffix.lower() == ".tsv" else ",")
    required = {"audio", "transcript", "license", "source"}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f"{path} is missing required columns: {sorted(missing)}")
    accepted, rejected = [], []
    limit = cfg.metadata_limit or len(table)
    for index, row in table.head(limit).iterrows():
        spec = SourceSpec(str(row["source"]), "local", None, str(row.get("domain", "user_supplied")), str(row["license"]), text_fields=("transcript",))
        record = row.to_dict() | {"id": row.get("sample_id", f"row-{index}"), "speaker_id": row.get("speaker_id", "unknown")}
        good, bad, _ = canonical_record(spec, str(row.get("split", "train")), record, int(index), cfg.audit_mode == "full")
        (accepted if good else rejected).append(good or bad)
    return accepted, rejected


for local_path in map(Path, cfg.local_manifest_paths):
    if local_path.exists():
        good, bad = scan_local_manifest(local_path)
        accepted_rows.extend(good)
        rejected_rows.extend(bad)


def spell4wiki_manifest(limit: int | None) -> tuple[list[dict], list[dict]]:
    endpoint = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query", "format": "json", "formatversion": 2,
        "generator": "categorymembers", "gcmtitle": "Category:Files uploaded by spell4wiki in dag",
        "gcmtype": "file", "gcmlimit": "max", "prop": "imageinfo",
        "iiprop": "url|sha1|mime|extmetadata",
    }
    accepted, rejected, continuation = [], [], {}
    while limit is None or len(accepted) + len(rejected) < limit:
        response = requests.get(endpoint, params=params | continuation, timeout=60)
        response.raise_for_status()
        payload = response.json()
        for page in payload.get("query", {}).get("pages", []):
            title = page.get("title", "")
            info = (page.get("imageinfo") or [{}])[0]
            metadata = info.get("extmetadata") or {}
            license_name = ((metadata.get("LicenseShortName") or {}).get("value") or "unknown").lower()
            filename = re.sub(r"^File:", "", title, flags=re.I)
            transcript = re.sub(r"\.(ogg|oga|wav|mp3)$", "", filename, flags=re.I)
            transcript = re.sub(r"^dag[-_ ]", "", transcript, flags=re.I).replace("_", " ").strip()
            admitted_for_audio = cfg.audit_mode == "full" and cfg.allow_network_audio_downloads
            row = {
                "sample_id": str(page.get("pageid", info.get("sha1", filename))),
                "source": "spell4wiki_commons", "source_version": "wikimedia-api",
                "audio_locator": info.get("url", ""), "transcript_raw": strict_text(transcript),
                "transcript_eval": normalize_eval(transcript), "speaker_id": "unknown",
                "duration_s": np.nan, "domain": "scripted_read",
                "split": "external_test" if admitted_for_audio else "candidate_external",
                "license": license_name, "audio_hash": info.get("sha1", ""),
                "is_pseudo": False, "pseudo_confidence": np.nan,
            }
            if license_name == "unknown" or not row["audio_locator"]:
                rejected.append(row | {"reason": "license_or_audio_url_missing"})
            elif admitted_for_audio:
                try:
                    array, sample_rate, _ = audio_payload(row["audio_locator"])
                    array = canonical_audio(array, sample_rate)
                    row["duration_s"] = len(array) / 16_000
                    row["audio_hash"] = exact_audio_hash(array)
                    if not (cfg.min_duration_s <= row["duration_s"] <= cfg.max_duration_s):
                        rejected.append(row | {"reason": "duration_out_of_range"})
                    else:
                        accepted.append(row)
                except Exception as exc:
                    rejected.append(row | {"reason": "audio_decode_failed", "error": f"{type(exc).__name__}: {exc}"})
            else:
                accepted.append(row)
            if limit is not None and len(accepted) + len(rejected) >= limit:
                break
        continuation = payload.get("continue") or {}
        if not continuation:
            break
    return accepted, rejected


spell_limit = None if cfg.audit_mode == "full" else cfg.spell4wiki_quick_limit
try:
    good, bad = spell4wiki_manifest(spell_limit)
    accepted_rows.extend(good)
    rejected_rows.extend(bad)
    source_reports.append({
        "name": "spell4wiki_commons", "status": "available" if good else "rejected",
        "domain": "scripted_read", "license": "per-file Commons license",
        "rows_scanned": len(good) + len(bad),
        "admission": "external_test" if cfg.audit_mode == "full" and cfg.allow_network_audio_downloads else "candidate_external",
    })
    print(f"Spell4Wiki metadata: accepted={len(good):,}, rejected={len(bad):,}")
except Exception as exc:
    source_reports.append({"name": "spell4wiki_commons", "status": "unavailable", "error": f"{type(exc).__name__}: {exc}"})
    print(f"Spell4Wiki unavailable: {exc}")

## 6. Duplicate, transcript, and split-leakage audit

In [ ]:
accepted_df = pd.DataFrame(accepted_rows)
rejected_df = pd.DataFrame(rejected_rows, columns=list(dict.fromkeys(CANONICAL_COLUMNS + ["reason", "error", "words_per_s"])))
near_df = pd.DataFrame(near_hash_rows, columns=["sample_id", "source", "near_hash"])
for column in CANONICAL_COLUMNS:
    if column not in accepted_df:
        accepted_df[column] = pd.Series(dtype="object")
accepted_df = accepted_df[CANONICAL_COLUMNS + [column for column in accepted_df.columns if column not in CANONICAL_COLUMNS]]


def is_official_waxal_speaker_overlap(group: pd.DataFrame) -> bool:
    '''WAXAL's published ASR splits are topic-level and reuse speakers.'''
    official_splits = {"train", "validation", "test"}
    return bool(
        group["source"].eq("waxal_dag_asr").all()
        and set(group["split"].unique()).issubset(official_splits)
        and ("split_policy" not in group or group["split_policy"].eq("official").all())
    )


def overlap_report(table: pd.DataFrame) -> pd.DataFrame:
    findings = []
    supervised = table[table["split"].isin(["train", "validation", "test", "external_test"])]
    for key in ("speaker_id", "audio_hash"):
        if key not in supervised or supervised.empty:
            continue
        clean = supervised[supervised[key].fillna("").astype(str).ne("")]
        if key == "speaker_id":
            clean = clean[clean[key] != "unknown"]
        for value, group in clean.groupby(key):
            splits = sorted(group["split"].unique())
            if len(splits) > 1:
                official_waxal = key == "speaker_id" and is_official_waxal_speaker_overlap(group)
                blocking = not official_waxal
                findings.append({
                    "kind": "waxal_official_topic_split_speaker_overlap" if official_waxal else f"{key}_overlap",
                    "value": value,
                    "sources": ",".join(sorted(group["source"].astype(str).unique())),
                    "splits": ",".join(splits),
                    "rows": len(group),
                    "severity": "warning" if official_waxal else "error",
                    "blocking": blocking,
                    "note": (
                        "Published WAXAL topic-level protocol; retained for benchmark comparability."
                        if official_waxal else "Must be resolved before full training."
                    ),
                })
    return pd.DataFrame(
        findings,
        columns=["kind", "value", "sources", "splits", "rows", "severity", "blocking", "note"],
    )


leakage_df = overlap_report(accepted_df)
blocking_leakage_df = leakage_df[leakage_df["blocking"].fillna(True)] if not leakage_df.empty else leakage_df
waxal_protocol_overlap_df = (
    leakage_df[leakage_df["kind"] == "waxal_official_topic_split_speaker_overlap"]
    if not leakage_df.empty else leakage_df
)
exact_duplicates = accepted_df[accepted_df["audio_hash"].fillna("").astype(str).ne("")].groupby("audio_hash").filter(lambda group: len(group) > 1)
transcript_duplicates = accepted_df[accepted_df["transcript_eval"].astype(str).ne("")].groupby("transcript_eval").filter(lambda group: len(group) > 1)
near_duplicates = near_df.groupby("near_hash").filter(lambda group: len(group) > 1) if not near_df.empty else near_df

inventory_rows = []
if not accepted_df.empty:
    for (source, split), group in accepted_df.groupby(["source", "split"], dropna=False):
        durations = pd.to_numeric(group["duration_s"], errors="coerce")
        inventory_rows.append({
            "source": source, "split": split, "rows_scanned": len(group),
            "decoded_rows": int(durations.notna().sum()), "decoded_hours": float(durations.sum() / 3600),
            "speakers": int(group["speaker_id"].replace("unknown", np.nan).nunique()),
            "transcript_coverage": float(group["transcript_raw"].astype(bool).mean()),
            "license_values": ",".join(sorted(group["license"].dropna().astype(str).unique())),
        })
inventory_df = pd.DataFrame(inventory_rows)
display(inventory_df)
print({
    "accepted": len(accepted_df), "rejected": len(rejected_df),
    "exact_duplicate_rows": len(exact_duplicates), "near_duplicate_rows": len(near_duplicates),
    "repeated_transcript_rows": len(transcript_duplicates),
    "reported_split_overlaps": len(leakage_df),
    "blocking_split_leaks": len(blocking_leakage_df),
    "waxal_protocol_speaker_overlaps": len(waxal_protocol_overlap_df),
})

## 7. Export immutable audit artifacts and readiness state

In [ ]:
def write_table(table: pd.DataFrame, stem: str) -> None:
    if len(table.columns) == 0:
        table = pd.DataFrame({"_empty": pd.Series(dtype="boolean")})
    table.to_csv(ARTIFACT_DIR / f"{stem}.csv", index=False)
    table.to_parquet(ARTIFACT_DIR / f"{stem}.parquet", index=False)


write_table(pd.DataFrame(source_reports), "source_status")
write_table(inventory_df, "corpus_inventory")
write_table(accepted_df, "accepted_manifest")
write_table(rejected_df, "rejected_samples")
write_table(leakage_df, "split_leakage_report")
write_table(exact_duplicates, "exact_audio_duplicates")
write_table(near_duplicates, "near_audio_duplicates")
write_table(transcript_duplicates, "repeated_transcripts")

environment = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "audit_config": asdict(cfg) | {"hf_token": "<redacted>"},
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("torch", "transformers", "datasets", "accelerate", "huggingface_hub", "jiwer", "librosa", "soundfile")
        if importlib.util.find_spec(name)
    },
}
(ARTIFACT_DIR / "environment_manifest.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")

waxal_supervised = accepted_df[
    accepted_df["source"].eq("waxal_dag_asr")
    & accepted_df["split"].isin(["train", "validation", "test"])
]
waxal_required_splits_present = {"train", "validation", "test"}.issubset(
    set(waxal_supervised["split"].unique())
)

readiness = {
    "audit_mode": cfg.audit_mode,
    "release_ready": bool(
        cfg.audit_mode == "full"
        and not accepted_df.empty
        and waxal_required_splits_present
        and blocking_leakage_df.empty
        and accepted_df[accepted_df["split"].isin(["train", "validation", "test", "external_test"])]["duration_s"].notna().all()
        and accepted_df[accepted_df["split"].isin(["train", "validation", "test", "external_test"])]["audio_hash"].astype(bool).all()
    ),
    "sealed_test_rows": int((accepted_df["split"] == "test").sum()) if not accepted_df.empty else 0,
    "waxal_required_splits_present": waxal_required_splits_present,
    "enabled_sources": list(cfg.enabled_sources),
    "warnings": [],
}
if cfg.audit_mode != "full":
    readiness["warnings"].append("Quick mode is diagnostic only; run full mode before full training.")
if not waxal_required_splits_present:
    readiness["warnings"].append("WAXAL train/validation/test are required for supervised training.")
if not waxal_protocol_overlap_df.empty:
    readiness["warnings"].append(
        "WAXAL official topic-level splits reuse speakers. Official metrics remain benchmark-comparable but are not speaker-independent."
    )
if not blocking_leakage_df.empty:
    readiness["warnings"].append("Cross-split speaker/audio leakage must be resolved.")
(ARTIFACT_DIR / "readiness.json").write_text(json.dumps(readiness, indent=2), encoding="utf-8")
print(json.dumps(readiness, indent=2))
print(f"Artifacts written to {ARTIFACT_DIR}")

## 8. Strict and minimally normalized ASR metrics

In [ ]:
import jiwer


def asr_metrics(references: list[str], hypotheses: list[str]) -> dict[str, float]:
    if len(references) != len(hypotheses):
        raise ValueError(f"Reference/hypothesis length mismatch: {len(references)} != {len(hypotheses)}")
    if not references:
        raise ValueError("At least one reference is required")
    strict_refs = [strict_text(value) for value in references]
    strict_hyps = [strict_text(value) for value in hypotheses]
    norm_refs = [normalize_eval(value) for value in references]
    norm_hyps = [normalize_eval(value) for value in hypotheses]
    return {
        "strict_wer": float(jiwer.wer(strict_refs, strict_hyps)),
        "strict_cer": float(jiwer.cer(strict_refs, strict_hyps)),
        "normalized_wer": float(jiwer.wer(norm_refs, norm_hyps)),
        "normalized_cer": float(jiwer.cer(norm_refs, norm_hyps)),
        "utterances": len(references),
    }


metric_self_test = asr_metrics(["N nyɛla bɛ."], ["n nyɛla bɛ"])
assert metric_self_test["normalized_wer"] == 0.0
print("Metric self-test passed:", metric_self_test)

## 9. Optional comparable baselines

Turn on a GPU, set `cfg.run_baselines = True`, and rerun this cell. Both models
receive the same frozen, filtered WAXAL test rows. The base model is evaluated
for context; `waxal-benchmarking/whisper-small-waxal-dag` is the 34.0% WER /
11.9% CER public reference. No language token is forced because Whisper has no
native Dagbani language token.

In [ ]:
def iter_waxal_benchmark(limit: int | None) -> Iterator[dict[str, Any]]:
    stream = load_dataset("google/WaxalNLP", "dag_asr", split="test", streaming=True, token=cfg.hf_token)
    yielded = 0
    for record in stream:
        array, sample_rate, _ = audio_payload(record["audio"])
        array = canonical_audio(array, sample_rate)
        reference = strict_text(record.get("transcription", ""))
        duration = len(array) / 16_000
        words_per_s = len(normalize_eval(reference).split()) / max(duration, 1e-6)
        if duration < cfg.waxal_benchmark_min_duration_s or words_per_s < cfg.waxal_benchmark_min_words_per_s:
            continue
        yield {"id": str(record.get("id", yielded)), "audio": array, "reference": reference, "duration_s": duration}
        yielded += 1
        if limit is not None and yielded >= limit:
            break


def benchmark_model(model_id: str, rows: list[dict[str, Any]]) -> tuple[dict, pd.DataFrame]:
    import torch
    from transformers import pipeline
    device = 0 if torch.cuda.is_available() else -1
    dtype = torch.float16 if device >= 0 else torch.float32
    recognizer = pipeline("automatic-speech-recognition", model=model_id, device=device, torch_dtype=dtype)
    predictions = []
    started = time.time()
    for index, row in enumerate(rows, 1):
        result = recognizer({"array": row["audio"], "sampling_rate": 16_000}, generate_kwargs={"task": "transcribe"})
        predictions.append(strict_text(result["text"]))
        if index % 25 == 0:
            print(f"{model_id}: {index}/{len(rows)} rows, {time.time() - started:.1f}s")
    references = [row["reference"] for row in rows]
    metrics = asr_metrics(references, predictions) | {"model_id": model_id, "elapsed_s": time.time() - started}
    table = pd.DataFrame({"sample_id": [row["id"] for row in rows], "reference": references, "hypothesis": predictions})
    return metrics, table


baseline_metrics = []
if cfg.run_baselines:
    eval_rows = list(iter_waxal_benchmark(cfg.baseline_limit))
    if not eval_rows:
        raise RuntimeError("No rows passed the WAXAL benchmark filter")
    for model_name in ("openai/whisper-small", "waxal-benchmarking/whisper-small-waxal-dag"):
        metrics, predictions = benchmark_model(model_name, eval_rows)
        baseline_metrics.append(metrics)
        predictions.to_parquet(ARTIFACT_DIR / f"baseline_{model_name.replace('/', '__')}.parquet", index=False)
        print(json.dumps(metrics, indent=2))
else:
    print("Baselines skipped. This preserves GPU quota during data auditing.")
(ARTIFACT_DIR / "baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")

## 10. Publish the manifest and audit evidence to a private Hugging Face dataset

In [ ]:
if cfg.publish_artifacts:
    from huggingface_hub import HfApi
    api = HfApi(token=cfg.hf_token)
    api.create_repo(cfg.manifest_repo_id, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(
        repo_id=cfg.manifest_repo_id,
        repo_type="dataset",
        folder_path=str(ARTIFACT_DIR),
        path_in_repo="phase1",
        commit_message=f"Dagbani ASR Phase 1 {cfg.audit_mode} audit",
    )
    print(f"Published privately to https://huggingface.co/datasets/{cfg.manifest_repo_id}")
else:
    print("Publishing disabled. Inspect all artifacts before setting publish_artifacts=True.")